# Практика 1. Контейнер, кадры, PTS, CRF и GOP

## 0. Подготовка среды
В Colab `ffmpeg` обычно уже установлен. Python-пакет `av` даёт доступ к декодеру FFmpeg из Python. Остальные библиотеки нужны для таблиц, вычислений и графиков.


In [ ]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "av", "pandas", "matplotlib", "numpy"],
    check=True,
)
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.PIPE, text=True)
print("Среда готова")


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import shutil
import subprocess
import time
import urllib.request
from collections import Counter
from fractions import Fraction
from pathlib import Path

import av
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Video, display

WORKDIR = Path("/content/video_io_practice")
WORKDIR.mkdir(parents=True, exist_ok=True)
print("Рабочая папка:", WORKDIR)


In [ ]:
SOURCE_URL = "https://download.blender.org/peach/trailer/trailer_1080p.mov"
downloaded_path = WORKDIR / "trailer_1080p.mov"
sample_path = WORKDIR / "sample.mp4"

if not sample_path.exists():
    try:
        print("Скачиваем открытое видео…")
        urllib.request.urlretrieve(SOURCE_URL, downloaded_path)
        subprocess.run([
            "ffmpeg", "-y", "-ss", "5", "-i", str(downloaded_path), "-t", "16",
            "-vf", "scale=640:-2", "-c:v", "libx264", "-preset", "veryfast",
            "-crf", "18", "-pix_fmt", "yuv420p", "-an", str(sample_path),
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except Exception as exc:
        print("Загрузка не удалась, создаём синтетический ролик:", exc)
        subprocess.run([
            "ffmpeg", "-y", "-f", "lavfi", "-i",
            "testsrc2=size=640x360:rate=30:duration=16",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
            "-pix_fmt", "yuv420p", str(sample_path),
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print(f"Файл: {sample_path} ({sample_path.stat().st_size / 2**20:.1f} MiB)")
display(Video(str(sample_path), embed=True, width=640))



1. `.mp4` — это кодек или контейнер?
2. Можно ли по расширению `.mp4` однозначно узнать кодек?



## 1. Метаданные и потоки контейнера

`ffprobe` читает структуру контейнера, не декодируя всё видео.


In [ ]:
def ffprobe_json(path: str | Path, *, show_frames: bool = False) -> dict:
    command = ["ffprobe", "-v", "error", "-print_format", "json"]
    if show_frames:
        command += ["-show_frames"]
    else:
        command += ["-show_format", "-show_streams"]
    command += [str(path)]
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    return json.loads(result.stdout)

probe = ffprobe_json(sample_path)
print("Число потоков:", len(probe["streams"]))
print(json.dumps(probe["streams"][0], ensure_ascii=False, indent=2)[:3000])


In [ ]:
def video_summary(path: str | Path) -> dict:
    data = ffprobe_json(path)
    stream = next(s for s in data["streams"] if s["codec_type"] == "video")
    fmt = data["format"]

    rate = stream.get("avg_frame_rate", "0/0")
    fps = float(Fraction(rate)) if rate != "0/0" else float("nan")
    duration = float(stream.get("duration") or fmt.get("duration") or 0)
    bitrate = int(stream.get("bit_rate") or fmt.get("bit_rate") or 0)
    return {
        "file": Path(path).name,
        "container": fmt.get("format_long_name"),
        "codec": stream.get("codec_name"),
        "profile": stream.get("profile"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "pix_fmt": stream.get("pix_fmt"),
        "fps": fps,
        "duration_s": duration,
        "bitrate_Mbit_s": bitrate / 1e6,
        "nb_frames": int(stream["nb_frames"]) if stream.get("nb_frames") else None,
    }

meta = video_summary(sample_path)
display(pd.DataFrame([meta]).T.rename(columns={0: "значение"}))


### Задание 1 — чтение метаданных (5 минут)

Найдите в `meta`:

- кодек и профиль;
- разрешение и pixel format;
- fps и длительность;
- ожидаемое число кадров как `fps × duration`.

Сравните ожидаемое число кадров с `nb_frames`. Почему значения могут немного различаться?


In [ ]:
expected_frames = None
print("expected_frames =", expected_frames)


<details>
<summary><b>Решение задания 1</b></summary>

```python
expected_frames = meta["fps"] * meta["duration_s"]
print(f"Ожидаем примерно {expected_frames:.1f} кадров")
print("nb_frames из контейнера:", meta["nb_frames"])
```

</details>


## 2. Декодирование кадров и временные метки

Кадр имеет индекс, но настоящая временная координата — **PTS** (presentation timestamp):

$$
t_i = \mathrm{PTS}_i \cdot \mathrm{time\_base}.
$$

Для CFR соседние интервалы почти одинаковы. Для VFR они заметно различаются. В ML-пайплайне индекс кадра нельзя без проверки считать временем.


In [ ]:
def read_frame_times(path: str | Path, limit: int | None = None) -> np.ndarray:
    times = []
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        for frame in container.decode(stream):
            if frame.pts is not None:
                times.append(float(frame.pts * stream.time_base))
            if limit is not None and len(times) >= limit:
                break
    return np.asarray(times)

pts = read_frame_times(sample_path, limit=240)
dt = np.diff(pts)
print(f"Кадров изучено: {len(pts)}")
print(f"Δt: mean={dt.mean():.6f}, min={dt.min():.6f}, max={dt.max():.6f} с")
print(f"fps по среднему Δt: {1 / dt.mean():.3f}")
print(f"max/min: {dt.max() / dt.min():.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(pts[:120], marker=".", ms=3)
axes[0].set(title="PTS первых кадров", xlabel="Индекс кадра", ylabel="Время, с")
axes[0].grid(alpha=0.3)
axes[1].hist(dt * 1000, bins=20)
axes[1].set(title="Распределение межкадровых интервалов", xlabel="Δt, мс")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Задание 2 — классифицируйте CFR/VFR

Реализуйте функцию `classify_rate(times, tolerance=0.01)`. Считайте видео CFR, если относительный разброс межкадровых интервалов

$$
\frac{\max \Delta t-\min \Delta t}{\mathrm{median}\,\Delta t}
$$

не превышает `tolerance`. Проверьте функцию на `pts`.


In [ ]:
def classify_rate(times: np.ndarray, tolerance: float = 0.01) -> str:
    # TODO
    return

print(classify_rate(pts))


<details>
<summary><b>Решение задания 2</b></summary>

```python
def classify_rate(times: np.ndarray, tolerance: float = 0.01) -> str:
    deltas = np.diff(times)
    if len(deltas) == 0 or np.median(deltas) <= 0:
        return "недостаточно данных"
    spread = (deltas.max() - deltas.min()) / np.median(deltas)
    return "CFR" if spread <= tolerance else "VFR"

print(classify_rate(pts))
```

</details>


## 3. Извлечение и визуализация кадров

PyAV декодирует видеокадр в объект `VideoFrame`. Для matplotlib преобразуем его в RGB-массив формы `(H, W, 3)`.


In [ ]:
wanted = {0, 90, 180, 270}
selected = []
with av.open(str(sample_path)) as container:
    for index, frame in enumerate(container.decode(video=0)):
        if index in wanted:
            selected.append((index, frame.time, frame.to_ndarray(format="rgb24")))
        if index >= max(wanted):
            break

fig, axes = plt.subplots(1, len(selected), figsize=(16, 4))
for ax, (index, frame_time, image) in zip(axes, selected):
    ax.imshow(image)
    ax.set_title(f"#{index}, t={frame_time:.2f} с\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. RGB, YCbCr и chroma subsampling

Кодеки обычно хранят яркость `Y` точнее, чем цветоразностные компоненты `Cb` и `Cr`. Формат `yuv420p` означает, что цвет хранится с половинным разрешением по каждой пространственной оси. На блок 2×2 приходится четыре Y, но только по одному Cb и Cr.



In [ ]:
image = selected[len(selected) // 2][2].astype(np.float32) / 255.0
r, g, b = image[..., 0], image[..., 1], image[..., 2]
y = 0.2126 * r + 0.7152 * g + 0.0722 * b
cb = (b - y) / 1.8556
cr = (r - y) / 1.5748

cb_420 = cb[::2, ::2]
cr_420 = cr[::2, ::2]
cb_up = np.repeat(np.repeat(cb_420, 2, axis=0), 2, axis=1)[: y.shape[0], : y.shape[1]]
cr_up = np.repeat(np.repeat(cr_420, 2, axis=0), 2, axis=1)[: y.shape[0], : y.shape[1]]

r2 = y + 1.5748 * cr_up
b2 = y + 1.8556 * cb_up
g2 = (y - 0.2126 * r2 - 0.0722 * b2) / 0.7152
reconstructed = np.clip(np.stack([r2, g2, b2], axis=-1), 0, 1)
error = np.abs(image - reconstructed)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(image); axes[0].set_title("Исходный RGB")
axes[1].imshow(reconstructed); axes[1].set_title("Имитация 4:2:0")
axes[2].imshow(np.clip(error * 10, 0, 1)); axes[2].set_title("|Ошибка| ×10")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("MAE:", float(error.mean()))


### Задание 3 — цена одного сырого кадра

Для текущего разрешения вычислите:

1. число байт кадра RGB24 (`3 байта/пиксель`);
2. число байт кадра YUV 4:2:0 8-bit (`1.5 байта/пиксель`);
3. экономию в процентах.


In [ ]:
width, height = meta["width"], meta["height"]
rgb_bytes = None
yuv420_bytes = None
saving_percent = None
print(rgb_bytes, yuv420_bytes, saving_percent)


<details>
<summary><b>Решение задания 3</b></summary>

```python
width, height = meta["width"], meta["height"]
rgb_bytes = width * height * 3
yuv420_bytes = width * height * 1.5
saving_percent = (1 - yuv420_bytes / rgb_bytes) * 100
print(f"RGB24: {rgb_bytes:,.0f} байт")
print(f"YUV420: {yuv420_bytes:,.0f} байт")
print(f"Экономия: {saving_percent:.0f}%")
```

</details>


## 5. Сырой и фактический битрейт

Для декодированного RGB24:

$$
B_{raw}=W\cdot H\cdot3\cdot8\cdot fps.
$$

Фактический средний битрейт файла оценим как размер в битах, делённый на длительность. Он включает контейнер и все потоки; в нашем рабочем файле аудио отключено.


In [ ]:
file_bits = sample_path.stat().st_size * 8
raw_bitrate = meta["width"] * meta["height"] * 3 * 8 * meta["fps"]
actual_bitrate = file_bits / meta["duration_s"]
compression_ratio = raw_bitrate / actual_bitrate

bitrate_table = pd.DataFrame({
    "показатель": ["Сырой RGB24", "Фактический файл", "Степень сжатия"],
    "значение": [f"{raw_bitrate / 1e6:.1f} Мбит/с", f"{actual_bitrate / 1e6:.2f} Мбит/с", f"×{compression_ratio:.1f}"],
})
display(bitrate_table)


## 6. Эксперимент с CRF

CRF управляет качеством x264: меньше число — выше качество и больше файл. Это не целевой битрейт; сложные сцены получают больше битов, простые — меньше.




In [ ]:
crf_dir = WORKDIR / "crf"
crf_dir.mkdir(exist_ok=True)
crf_values = [18, 23, 28, 35]

for crf in crf_values:
    output = crf_dir / f"sample_crf{crf}.mp4"
    if not output.exists():
        subprocess.run([
            "ffmpeg", "-y", "-i", str(sample_path), "-c:v", "libx264",
            "-preset", "veryfast", "-crf", str(crf), "-pix_fmt", "yuv420p",
            "-an", str(output),
        ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(crf, f"{output.stat().st_size / 2**20:.2f} MiB")


In [ ]:
def frame_at_index(path: str | Path, target: int) -> np.ndarray:
    with av.open(str(path)) as container:
        for index, frame in enumerate(container.decode(video=0)):
            if index == target:
                return frame.to_ndarray(format="rgb24")
    raise IndexError(target)

def psnr(reference: np.ndarray, candidate: np.ndarray) -> float:
    mse = np.mean((reference.astype(np.float64) - candidate.astype(np.float64)) ** 2)
    return float("inf") if mse == 0 else 10 * math.log10(255**2 / mse)

frame_index = min(180, int((meta["nb_frames"] or 300) // 2))
reference = frame_at_index(sample_path, frame_index)
h, w = reference.shape[:2]
crop = (slice(h // 4, 3 * h // 4), slice(w // 4, 3 * w // 4))

rows = []
fig, axes = plt.subplots(1, len(crf_values), figsize=(16, 4))
for ax, crf in zip(axes, crf_values):
    path = crf_dir / f"sample_crf{crf}.mp4"
    image_crf = frame_at_index(path, frame_index)
    score = psnr(reference, image_crf)
    size_mib = path.stat().st_size / 2**20
    rows.append({"CRF": crf, "size_MiB": size_mib, "PSNR_dB": score})
    ax.imshow(image_crf[crop])
    ax.set_title(f"CRF {crf}\n{size_mib:.2f} MiB, {score:.1f} dB")
    ax.axis("off")
plt.tight_layout(); plt.show()
crf_results = pd.DataFrame(rows)
display(crf_results)


In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(crf_results["CRF"], crf_results["size_MiB"], "o-", color="tab:blue")
ax1.set(xlabel="CRF", ylabel="Размер, MiB")
ax2 = ax1.twinx()
ax2.plot(crf_results["CRF"], crf_results["PSNR_dB"], "s--", color="tab:red")
ax2.set_ylabel("PSNR, dB")
ax1.grid(alpha=0.3)
plt.title("Компромисс размер–качество")
plt.show()


### Задание 4 — интерпретация CRF

По таблице и кропам ответьте:

1. Во сколько раз версия CRF 35 меньше версии CRF 18?
2. Монотонно ли меняется PSNR?
3. Где артефакты заметны раньше: на гладких областях, контурах или текстурах?
4. Почему PSNR не является полной заменой человеческой оценки качества?

Впишите вычисление коэффициента в ячейку.


In [ ]:
size_18 = float(crf_results.loc[crf_results.CRF == 18, "size_MiB"].iloc[0])
size_35 = float(crf_results.loc[crf_results.CRF == 35, "size_MiB"].iloc[0])
size_ratio_18_to_35 = None
print("CRF 18 / CRF 35 =", size_ratio_18_to_35)


<details>
<summary><b>Решение задания 4</b></summary>

```python
size_ratio_18_to_35 = size_18 / size_35
print(f"CRF 18 больше CRF 35 в {size_ratio_18_to_35:.2f} раза")
```

Обычно размер и PSNR убывают с ростом CRF, но точные значения зависят от сцены и настроек кодека. Артефакты хорошо видны на границах и мелких текстурах, а banding — на плавных градиентах. PSNR измеряет пиксельную ошибку, но не моделирует восприятие, временное мерцание и значимость разных деталей.
</details>


## 7. I/P/B-кадры и GOP

- **I-frame** декодируется самостоятельно и служит точкой входа для seek.
- **P-frame** предсказывается по предыдущим опорным кадрам.
- **B-frame** может использовать предыдущие и будущие кадры.
- **GOP** — расстояние между соседними I-кадрами (в практическом анализе часто измеряют числом кадров).


In [ ]:
def frame_table(path: str | Path) -> pd.DataFrame:
    command = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "frame=pict_type,pkt_size,best_effort_timestamp_time,key_frame",
        "-of", "json", str(path),
    ]
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    frames = json.loads(result.stdout)["frames"]
    table = pd.DataFrame(frames)
    table["pkt_size"] = pd.to_numeric(table["pkt_size"], errors="coerce")
    table["best_effort_timestamp_time"] = pd.to_numeric(
        table["best_effort_timestamp_time"], errors="coerce"
    )
    return table

frames_df = frame_table(sample_path)
display(frames_df.head(20))
display(frames_df.groupby("pict_type")["pkt_size"].agg(["count", "mean", "median"]))

i_positions = np.flatnonzero(frames_df["pict_type"].eq("I").to_numpy())
gop_lengths = np.diff(i_positions)
print("Первые позиции I-кадров:", i_positions[:10])
print("Длины GOP:", gop_lengths[:10])
if len(gop_lengths):
    print(f"Средний GOP: {gop_lengths.mean():.1f} кадров = {gop_lengths.mean()/meta['fps']:.2f} с")


In [ ]:
palette = {"I": "tab:red", "P": "tab:blue", "B": "tab:green"}
first = frames_df.head(150).copy()
colors = first["pict_type"].map(palette).fillna("gray")
plt.figure(figsize=(14, 4))
plt.bar(np.arange(len(first)), first["pkt_size"] / 1024, color=colors, width=1.0)
plt.xlabel("Индекс кадра")
plt.ylabel("Размер пакета, KiB")
plt.title("Красный — I, синий — P, зелёный — B")
plt.grid(axis="y", alpha=0.3)
plt.show()


### Задание 5 — статистика GOP

Вычислите:

- долю I-, P- и B-кадров в процентах;
- отношение среднего размера I-кадра к среднему размеру P-кадра;
- максимальную теоретическую задержку seek до нужного кадра в кадрах и секундах, если GOP постоянный.


In [ ]:
type_percent = None
i_to_p_ratio = None
worst_seek_frames = None
print(type_percent, i_to_p_ratio, worst_seek_frames)


<details>
<summary><b>Решение задания 5</b></summary>

```python
type_percent = frames_df["pict_type"].value_counts(normalize=True).mul(100)
means = frames_df.groupby("pict_type")["pkt_size"].mean()
i_to_p_ratio = means.get("I", np.nan) / means.get("P", np.nan)
worst_seek_frames = int(gop_lengths.max()) if len(gop_lengths) else None
print(type_percent)
print(f"I/P по среднему размеру: ×{i_to_p_ratio:.2f}")
if worst_seek_frames:
    print(f"До {worst_seek_frames} кадров, или {worst_seek_frames/meta['fps']:.2f} с")
```

Реальный seek зависит не только от GOP: важны индекс контейнера, кэш, скорость носителя и декодера. Но без декодирования зависимых кадров получить произвольный P/B-кадр нельзя.
</details>


## 8. Короткий GOP и стоимость случайного доступа

Создадим версию с I-кадром примерно каждую секунду (`-g fps`) и запретим x264 автоматически менять положение ключевых кадров. Затем сравним структуру и размеры.


In [ ]:
short_gop_path = WORKDIR / "sample_gop_1s.mp4"
gop_size = round(meta["fps"])
subprocess.run([
    "ffmpeg", "-y", "-i", str(sample_path), "-c:v", "libx264",
    "-preset", "veryfast", "-crf", "23", "-g", str(gop_size),
    "-keyint_min", str(gop_size), "-sc_threshold", "0", "-an", str(short_gop_path),
], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

short_frames = frame_table(short_gop_path)
short_i = np.flatnonzero(short_frames["pict_type"].eq("I").to_numpy())
print("GOP исходного файла:", gop_lengths[:10])
print("GOP новой версии:", np.diff(short_i)[:10])
print(f"Размер исходного: {sample_path.stat().st_size / 2**20:.2f} MiB")
print(f"Размер short-GOP: {short_gop_path.stat().st_size / 2**20:.2f} MiB")


## Дополнительное задание: контейнер или кодек?

Проведём два разных преобразования:

1. **Remux:** переложим тот же H.264-поток из MP4 в MKV без перекодирования (`-c copy`). Пиксели и кодек не должны измениться.
2. **Transcode:** декодируем H.264 и заново закодируем видео в VP9/WebM. Здесь изменятся кодек, размер и пиксельные значения.

Сначала сделаем remux и сравним хеши извлечённого элементарного H.264-потока.


In [ ]:
import hashlib

remux_path = WORKDIR / "sample_remux.mkv"
start = time.perf_counter()
subprocess.run([
    "ffmpeg", "-y", "-i", str(sample_path), "-map", "0:v:0",
    "-c", "copy", str(remux_path),
], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
remux_seconds = time.perf_counter() - start

def h264_stream_hash(path: str | Path) -> str:
    result = subprocess.run([
        "ffmpeg", "-v", "error", "-i", str(path), "-map", "0:v:0",
        "-c", "copy", "-bsf:v", "h264_mp4toannexb", "-f", "h264", "pipe:1",
    ], check=True, capture_output=True)
    return hashlib.sha256(result.stdout).hexdigest()

remux_comparison = pd.DataFrame([
    video_summary(sample_path),
    video_summary(remux_path),
])
display(remux_comparison)
source_hash = h264_stream_hash(sample_path)
remux_hash = h264_stream_hash(remux_path)
print(f"Время remux: {remux_seconds:.3f} с")
print("SHA-256 MP4:", source_hash)
print("SHA-256 MKV:", remux_hash)
print("Поток идентичен:", source_hash == remux_hash)


Теперь выполним настоящее перекодирование в VP9. Параметры CRF у разных кодеков **нельзя напрямую сравнивать как одну шкалу качества**, поэтому здесь мы фиксируем настройки, измеряем результат и не делаем универсальных выводов по одному ролику.


In [ ]:
vp9_path = WORKDIR / "sample_vp9.webm"
start = time.perf_counter()
subprocess.run([
    "ffmpeg", "-y", "-i", str(sample_path), "-c:v", "libvpx-vp9",
    "-crf", "32", "-b:v", "0", "-deadline", "good", "-cpu-used", "4",
    "-an", str(vp9_path),
], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
vp9_seconds = time.perf_counter() - start

vp9_meta = video_summary(vp9_path)
vp9_frame = frame_at_index(vp9_path, frame_index)
vp9_psnr = psnr(reference, vp9_frame)

codec_comparison = pd.DataFrame([
    {
        "variant": "H.264 / MP4",
        "codec": meta["codec"],
        "size_MiB": sample_path.stat().st_size / 2**20,
        "operation_s": np.nan,
        "PSNR_to_working_source_dB": float("inf"),
    },
    {
        "variant": "H.264 / MKV (remux)",
        "codec": video_summary(remux_path)["codec"],
        "size_MiB": remux_path.stat().st_size / 2**20,
        "operation_s": remux_seconds,
        "PSNR_to_working_source_dB": float("inf"),
    },
    {
        "variant": "VP9 / WebM (transcode)",
        "codec": vp9_meta["codec"],
        "size_MiB": vp9_path.stat().st_size / 2**20,
        "operation_s": vp9_seconds,
        "PSNR_to_working_source_dB": vp9_psnr,
    },
])
display(codec_comparison)


### Задание 6 — объясните результаты

1. Какие поля изменились после remux, а какие остались прежними?
2. Почему хеши H.264-потока совпадают, хотя файлы MP4 и MKV имеют разные байты и размеры?
3. Во сколько раз transcode в VP9 был медленнее remux?
4. Можно ли по одному ролику и произвольно выбранным CRF утверждать, что VP9 всегда лучше или хуже H.264?
5. Какую операцию следует выбрать, если нужно только сменить контейнер, не ухудшая качество?


In [ ]:
vp9_to_remux_time_ratio = None
vp9_to_h264_size_ratio = None
print("VP9 transcode / remux по времени:", vp9_to_remux_time_ratio)
print("VP9 / H.264 по размеру:", vp9_to_h264_size_ratio)


<details>
<summary><b>Решение задания 6</b></summary>

```python
vp9_to_remux_time_ratio = vp9_seconds / remux_seconds
vp9_to_h264_size_ratio = vp9_path.stat().st_size / sample_path.stat().st_size
print(f"Transcode медленнее remux примерно в {vp9_to_remux_time_ratio:.1f} раза")
print(f"Размер VP9 составляет {vp9_to_h264_size_ratio:.1%} от H.264")
```

При remux меняется контейнер и его служебные данные, но сжатый H.264-поток копируется без декодирования и повторного кодирования. Поэтому операция быстрая, качество не меняется, а хеш извлечённого потока совпадает. При transcode исходные кадры декодируются и снова кодируются с потерями. Одного ролика и одной пары настроек недостаточно для общего сравнения эффективности кодеков: нужен набор сцен и сопоставимое качество. Для простой смены контейнера выбирают `-c copy`.
</details>
